In this work, we use transformer model to integrate gene expression and TCR amino acid sequences

Getting gene data

In [1]:
# %matplotlib inline

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os 
import numpy as np

import pandas as pd
# import seaborn as sb
import matplotlib.pyplot as pl

import scanpy as sc

import anndata as ad

from scipy.sparse import csr_matrix
from matplotlib import rcParams
from matplotlib import colors

sc.settings.verbosity = 3


In [2]:
gene_TCR = ad.read_h5ad('../10Xdatasets/gex_merge_log1_5000_genes_all_peptides.h5ad')
gene_TCR

/home/phile/miniconda3/envs/integration/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 145479 × 5000
    obs: 'v_gene_TRA', 'v_gene_TRB', 'd_gene_TRA', 'd_gene_TRB', 'j_gene_TRA', 'j_gene_TRB', 'c_gene_TRA', 'c_gene_TRB', 'cdr3_TRA', 'cdr3_TRB', 'cdr3_nt_TRA', 'cdr3_nt_TRB', 'umis_TRA', 'umis_TRB', 'donor', 'cell_clono_cdr3_aa', 'cell_clono_cdr3_nt', 'CD3', 'CD19', 'CD45RA', 'CD4', 'CD8a', 'CD14', 'CD45RO', 'CD279_PD-1', 'IgG1', 'IgG2a', 'IgG2b', 'CD127', 'CD197_CCR7', 'HLA-DR', 'A0101_VTEHDTLLY_IE-1_CMV', 'A0201_KTWGQYWQV_gp100_Cancer', 'A0201_ELAGIGILTV_MART-1_Cancer', 'A0201_CLLWSFQTSA_Tyrosinase_Cancer', 'A0201_IMDQVPFSV_gp100_Cancer', 'A0201_SLLMWITQV_NY-ESO-1_Cancer', 'A0201_KVAELVHFL_MAGE-A3_Cancer', 'A0201_KVLEYVIKV_MAGE-A1_Cancer', 'A0201_CLLGTYTQDV_Kanamycin-B-dioxygenase', 'A0201_LLDFVRFMGV_EBNA-3B_EBV', 'A0201_LLMGTLGIVC_HPV-16E7_82-91', 'A0201_CLGGLLTMV_LMP-2A_EBV', 'A0201_YLLEMLWRL_LMP1_EBV', 'A0201_FLYALALLL_LMP2A_EBV', 'A0201_GILGFVFTL_Flu-MP_Influenza', 'A0201_GLCTLVAML_BMLF1_EBV', 'A0201_NLVPMVATV_pp65_CMV', 'A0201_I

In [3]:
gene = pd.DataFrame(gene_TCR.layers['raw_counts'].todense())
gene

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,5.0,1.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0
4,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145474,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
145475,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,5.0,3.0,0.0,0.0
145476,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
145477,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0


In [4]:
tcr_seq = gene_TCR.obs[['cdr3_TRA']]
tcr_seq

,cdr3_TRA
barcode,
AGGGTGAGTATTACCG-18,CALSEASGTYKYIF
CTTGGCTTCGTTGCCT-25,CAAILFGNEKLTF
ACGATACTCGCAGGCT-40,CALSADTGNQFYF
ACGCCAGTCATGTCTT-8,CAMNPAWGGATNKLIF
TTCTTAGCAAAGAATC-4,CAETALGNTGKLIF
...,...
GAAGCAGAGCAGGCTA-3,CAVEPLYGNKLVF
CAGTCCTTCATCACCC-8,CAATHASGYDKVIF
GACTACACACGGTAAG-3,CAVDLMKTSYDKVIF


In [5]:
import tensorflow as tf

2025-07-22 21:03:39.591312: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-22 21:03:39.795727: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-22 21:03:39.887798: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [6]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Reshape, Conv2D, Conv2DTranspose, Flatten, Dropout
from tensorflow.keras.initializers import HeNormal
# Define input layer
input_gex = Input(shape=(100,))
gex = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(input_gex)
gex = Reshape(target_shape=(8,8,1))(gex)

# Convolutional layers
gex = Conv2D(filters=64, kernel_size=3, strides=1, activation='relu', padding="same")(gex)
gex = Conv2D(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(gex)

gex = Flatten()(gex)
hidden_layer = Dense(units=225, activation='relu', kernel_initializer=HeNormal())(gex)

# Transposed Convolutional layers
tcr = Reshape(target_shape=(15,15,1))(hidden_layer)
tcr = Conv2DTranspose(filters=16, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
tcr = Conv2DTranspose(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
tcr = Dropout(rate=0.2)(tcr)
tcr = Flatten()(tcr)
tcr = Dense(units=150, activation='relu', kernel_initializer=HeNormal())(tcr)
tcr = Dense(units=130, activation='relu')(tcr)  # Ensure proper activation
tcr = Dense(units=121, activation='relu')(tcr)
# Define model
model = Model(inputs=input_gex, outputs=tcr)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0015),
              loss='mse')

# Check layer names
model.summary()

2025-07-22 21:03:42.184534: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-22 21:03:42.329306: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21210 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:17:00.0, compute capability: 8.6


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 100)]             0         
                                                                 
 dense (Dense)               (None, 64)                6464      
                                                                 
 reshape (Reshape)           (None, 8, 8, 1)           0         
                                                                 
 conv2d (Conv2D)             (None, 8, 8, 64)          640       
                                                                 
 conv2d_1 (Conv2D)           (None, 8, 8, 32)          18464     
                                                                 
 flatten (Flatten)           (None, 2048)              0         
                                                                 
 dense_1 (Dense)             (None, 225)               461025

In [7]:
# import tensorflow as tf
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Input, Dense, Reshape, Conv2D, Conv2DTranspose, Flatten
# from tensorflow.keras.initializers import HeNormal

# # Define input layer
# input_gex = Input(shape=(100,))
# gex = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(input_gex)
# gex = Reshape(target_shape=(8,8,1))(gex)

# # Convolutional layers
# gex = Conv2D(filters=64, kernel_size=3, strides=1, activation='relu', padding="same")(gex)
# gex = Conv2D(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(gex)

# gex = Flatten()(gex)
# hidden_layer = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(gex)

# # Transposed Convolutional layers
# tcr = Reshape(target_shape=(8,8,1))(hidden_layer)
# tcr = Conv2DTranspose(filters=16, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
# tcr = Conv2DTranspose(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)

# tcr = Flatten()(tcr)
# tcr = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(tcr)
# tcr = Dense(units=121)(tcr)  # Ensure proper activation

# # Define model
# model = Model(inputs=input_gex, outputs=tcr)
# model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
#               loss='mse')

# # Check layer names
# model.summary()


In [8]:
AE_tcr = pd.read_csv("../AE_emb_TRA_all_peptides_10X_all_donors.csv")
AE_tcr

,0,1,2,3,4,5,6,7,8,9,...,111,112,113,114,115,116,117,118,119,120
0,-316.207800,-800.64910,-320.83533,346.98510,705.439450,464.974000,809.040000,-895.359200,-1054.613000,63.288700,...,116.577480,1000.989600,194.82726,-46.40611,-925.15344,504.46783,242.50058,807.05420,-840.84436,563.784200
1,107.312775,-176.85265,102.58903,259.01474,-389.094970,-170.045550,578.626900,-56.362595,119.735520,-1107.644800,...,596.523200,114.596214,178.70210,-618.82056,-126.63640,694.53640,458.34180,163.17456,-900.83606,-564.017150
2,-682.544400,126.96287,-266.74478,811.06690,54.793247,1033.749500,-5.409876,-477.162020,-626.985600,107.843820,...,25.007103,342.110350,701.48700,-220.56151,-274.78030,223.89500,-241.35388,-33.82157,-857.79560,44.841785
3,-239.374860,-539.07450,575.21450,-95.55604,-700.538450,-136.053050,-1057.042000,104.907840,-1014.197700,359.407930,...,-398.675840,270.228850,-488.31903,38.05454,-748.61510,135.54909,-319.03033,407.83580,-370.18942,-125.665770
4,-610.030900,-91.12127,-39.60043,-118.94101,455.614140,164.932970,550.304100,-489.563350,6.282069,-311.035920,...,50.310820,-81.869070,1121.82230,378.41960,260.26187,643.58246,611.14950,1115.37130,-1020.05316,133.965320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145474,-74.493700,981.68530,-577.37530,80.77775,-110.932080,-38.066833,293.816070,61.911617,-337.861570,552.293460,...,382.998200,227.379090,154.56630,-429.43268,-259.98810,458.89886,763.64197,602.20220,-777.30910,-1238.829200
145475,-1575.272600,363.77660,-873.83280,562.79400,-473.024100,-72.573760,336.400300,-1090.651400,395.956420,50.373066,...,-234.910930,625.048600,776.61240,287.09317,225.88951,1206.85420,-217.31377,1682.24980,-1090.58060,598.500000
145476,480.570680,-144.17064,119.83445,589.88513,-349.911400,-180.624530,649.415160,95.300840,-695.461100,-119.363530,...,-441.681460,834.983200,751.47670,-261.79773,697.58650,-201.78462,1361.48830,-67.69755,-651.23334,-256.774750
145477,-1076.087000,497.47763,376.98120,179.86403,100.149270,-1056.312600,410.896270,-764.348200,615.423300,-425.644530,...,490.229200,867.736760,362.18200,-200.04040,540.37300,751.93810,-323.77700,1519.47960,-1185.24760,764.347050


In [9]:
import numpy as np
from sklearn.decomposition import NMF

# Generate random non-negative data
data = gene.to_numpy()

# Initialize the NMF model
n_components = 100
model_nmf = NMF(n_components=n_components, init='random', random_state=0)

# Fit the model to the data
W = model_nmf.fit_transform(data)
H = model_nmf.components_

# Display the results
print("Basis matrix (W):\n", W)
print("Coefficients matrix (H):\n", H)


KeyboardInterrupt: 

In [ ]:
W = pd.DataFrame(W)
W

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.227333,7.086625,0.466240,0.436161,0.334158,0.003627,0.198576,0.001280,0.004045,0.002781,...,0.004944,0.033126,0.143294,0.116318,0.053315,0.090684,0.018362,0.296472,0.013140,0.083483
1,1.316014,5.126111,0.649829,0.694511,0.023224,0.000310,0.057097,0.012929,0.008026,0.030475,...,0.007351,0.088509,0.556798,0.192283,0.042265,0.103002,0.001494,0.392443,0.000000,0.052590
2,0.000000,0.000000,0.405181,0.836287,4.124795,0.007945,0.000000,0.003714,0.002770,0.026438,...,0.006715,0.044787,0.544515,0.088714,0.143255,0.056269,0.018553,0.658247,0.050502,0.113769
3,0.000000,2.308417,0.119431,0.120097,3.985986,0.015477,0.000000,0.004406,0.025324,0.015539,...,0.009446,0.125159,0.000000,0.071700,0.156441,0.074255,0.014709,0.105227,0.006118,0.081632
4,0.000000,0.000000,0.546531,1.292965,3.482252,0.010140,0.046020,0.018085,0.022204,0.046295,...,0.061491,0.027266,0.298706,0.421875,0.362350,0.100456,0.010704,0.897931,0.000000,0.181201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145474,0.000000,0.000000,0.132730,0.000000,0.000000,0.000000,0.024647,0.000000,0.000000,0.001994,...,0.002062,0.000000,0.025164,0.052784,0.000000,0.000000,0.000000,0.171280,0.000000,0.066642
145475,0.000000,2.239055,0.186252,0.369426,0.800173,0.016219,0.026338,0.019682,0.022974,0.000450,...,0.002169,0.031499,0.040721,0.270443,0.118005,0.139055,0.019971,0.630590,0.052803,0.172927
145476,0.024406,0.937475,0.406066,0.069124,2.025419,0.009820,0.002730,0.004958,0.010678,0.029238,...,0.000000,0.024329,0.136838,0.133530,0.076204,0.023668,0.002195,0.451703,0.004854,0.133263
145477,0.000000,0.000000,0.244838,0.000000,0.000000,0.002488,0.039718,0.000000,0.001914,0.000000,...,0.000000,0.004401,0.046355,0.000000,0.011181,0.003384,0.009216,0.004487,0.000000,0.025856


In [ ]:

# es_callback = EarlyStopping(monitor= 'val_auc', patience=20, restore_best_weights=True)
reduce_learning_rate = tf.keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=5, monitor='loss', min_delta=0.01)

history = model.fit(W,AE_tcr, 
                epochs=1400, 
                batch_size=128, 
                shuffle = True
                # callbacks=[es_callback, checkpoint,reduce_learning_rate])
                # callbacks=[es_callback,reduce_learning_rate]
                )

Epoch 1/1400


2025-07-22 17:50:22.584791: I tensorflow/stream_executor/cuda/cuda_blas.cc:1614] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2025-07-22 17:50:23.270978: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8100
2025-07-22 17:50:24.091258: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


1137/1137 [==============================] - 13s 9ms/step - loss: 279383.5938
Epoch 2/1400
1137/1137 [==============================] - 10s 9ms/step - loss: 277509.1875
Epoch 3/1400
1137/1137 [==============================] - 11s 10ms/step - loss: 274531.1562
Epoch 4/1400
1137/1137 [==============================] - 11s 10ms/step - loss: 267725.0312
Epoch 5/1400
1137/1137 [==============================] - 11s 10ms/step - loss: 265712.2812
Epoch 6/1400
1137/1137 [==============================] - 11s 10ms/step - loss: 264239.3125
Epoch 7/1400
1137/1137 [==============================] - 11s 10ms/step - loss: 263086.3750
Epoch 8/1400
1137/1137 [==============================] - 10s 9ms/step - loss: 262111.7031
Epoch 9/1400
1137/1137 [==============================] - 11s 10ms/step - loss: 261307.1094
Epoch 10/1400
1137/1137 [==============================] - 10s 9ms/step - loss: 260619.1875
Epoch 11/1400
1137/1137 [==============================] - 10s 9ms/step - loss: 260122.8438
Epoc

KeyboardInterrupt: 

In [ ]:
for layer in model.layers:
    print(layer.name)

input_1
dense
reshape
conv2d
conv2d_1
flatten
dense_1
reshape_1
conv2d_transpose
conv2d_transpose_1
dropout
flatten_1
dense_2
dense_3
dense_4


In [ ]:
from tensorflow.keras.models import Model
latent_model = Model(inputs=input_gex, outputs=hidden_layer)


In [ ]:
model.predict( W.iloc[1:2])

1/1 [==============================] - 0s 231ms/step


array([[   0.      ,  282.87418 ,    0.      ,  179.37187 ,    0.      ,
           0.      ,    0.      ,    0.      ,    0.      ,    0.      ,
         226.20221 ,   10.472186,    0.      ,    0.      ,    0.      ,
         451.1031  ,  680.6808  ,  234.66826 ,    0.      ,  265.87424 ,
         255.60876 ,    0.      ,    0.      ,    0.      ,    0.      ,
           0.      ,    0.      ,    0.      , 1066.8148  ,    0.      ,
           0.      ,    0.      ,    0.      ,    0.      ,    0.      ,
         581.25586 ,    0.      ,    0.      ,    0.      ,    0.      ,
           0.      ,  769.2237  ,    0.      ,    0.      ,   16.353132,
           0.      ,    0.      ,   23.847935,    0.      ,    0.      ,
         366.39087 ,  374.7697  ,  323.41495 ,    8.949597,    0.      ,
           0.      ,    0.      ,  644.609   ,  183.93542 ,  257.91528 ,
           0.      ,  416.4306  ,    0.      ,   49.78854 ,  284.2461  ,
           0.      ,    0.      , 3711.6548  ,   83

In [ ]:
model.predict( W.iloc[4:5])

1/1 [==============================] - 0s 43ms/step


array([[   0.       ,  365.12427  ,    0.       ,    0.       ,
           0.       ,    0.       ,  419.0646   ,    0.       ,
           0.       ,    0.       ,    0.       ,    0.       ,
           0.       ,    0.       ,    0.       ,    0.       ,
         443.47708  ,    0.       ,    0.       ,  510.39874  ,
           0.       ,    0.       ,    0.       ,    0.       ,
           0.       ,    0.       ,    0.       ,  137.07036  ,
         424.77206  ,   21.36535  ,    0.       ,   76.86337  ,
         260.05426  ,    0.       ,  696.32     ,  341.5926   ,
         460.0225   ,  165.50032  ,    0.       ,    0.       ,
        1123.8145   ,  330.2831   ,    3.4851475,   54.808918 ,
           0.       ,    0.       ,  318.08087  ,    0.       ,
          27.86477  ,  546.44714  ,    0.       ,  223.28453  ,
        1512.1534   ,   84.55223  ,  377.29678  ,    0.       ,
           0.       ,  352.13083  ,    0.       ,    0.       ,
           0.       ,    0.       ,    0

In [ ]:
integration_pred = latent_model.predict( W)

4547/4547 [==============================] - 14s 3ms/step


In [ ]:
pd.DataFrame(integration_pred[1:50,1:50])

,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,0.0,0.000000,0.000000,73.942764,0.0,0.0,0.0,148.284988,0.000000,59.473297,...,0.0,77.397285,0.000000,0.0,133.583817,0.0,0.0,72.280869,0.0,0.0
1,0.0,0.000000,0.000000,83.532921,0.0,0.0,0.0,161.269333,0.000000,219.395828,...,0.0,0.000000,0.000000,0.0,161.774597,0.0,0.0,7.269670,0.0,0.0
2,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.918569,...,0.0,56.906204,0.000000,0.0,407.721161,0.0,0.0,0.000000,0.0,0.0
3,0.0,243.816971,0.000000,197.551590,0.0,0.0,0.0,212.702637,190.704697,325.734161,...,0.0,0.000000,297.092896,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0
4,0.0,220.942734,0.000000,81.349754,0.0,0.0,0.0,65.122330,0.000000,23.175999,...,0.0,0.000000,0.000000,0.0,155.669098,0.0,0.0,0.000000,0.0,0.0
5,0.0,0.000000,0.000000,205.058075,0.0,0.0,0.0,28.442026,0.000000,54.506237,...,0.0,338.641510,0.000000,0.0,354.915253,0.0,0.0,0.000000,0.0,0.0
6,0.0,0.000000,0.000000,64.408531,0.0,0.0,0.0,14.082086,0.000000,197.532669,...,0.0,56.651707,0.000000,0.0,11.822893,0.0,0.0,149.356995,0.0,0.0
7,0.0,189.587250,0.000000,85.999336,0.0,0.0,0.0,78.129402,0.000000,1.852733,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0
8,0.0,100.950172,0.000000,85.258110,0.0,0.0,0.0,0.000000,0.000000,366.110138,...,0.0,139.227905,111.221626,0.0,46.511890,0.0,0.0,0.000000,0.0,0.0
9,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,14.327515,0.000000,234.530472,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,170.058945,0.0,0.0


In [ ]:
integration_pred = integration_pred.reshape([integration_pred.shape[0],-1])

In [ ]:
integration_pred.shape

(145479, 225)

In [ ]:
integration_pred = integration_pred.reshape([integration_pred.shape[0],-1])
integration_pred.shape

(145479, 225)

In [ ]:
pd.DataFrame(integration_pred)

,0,1,2,3,4,5,6,7,8,9,...,215,216,217,218,219,220,221,222,223,224
0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,55.963898,59.682621,...,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,279.597900,0.000000
1,0.000000,0.0,0.000000,0.0,73.942764,0.0,0.0,0.0,148.284988,0.000000,...,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000
2,93.040886,0.0,0.000000,0.0,83.532921,0.0,0.0,0.0,161.269333,0.000000,...,49.709923,0.0,0.0,0.000000,0.0,0.000000,99.580719,0.0,27.306803,0.000000
3,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,...,0.000000,0.0,0.0,0.000000,0.0,0.000000,22.845297,0.0,174.727341,0.000000
4,0.000000,0.0,243.816971,0.0,197.551590,0.0,0.0,0.0,212.702637,190.704697,...,97.653908,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,205.183258,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145474,0.000000,0.0,155.781082,0.0,0.000000,0.0,0.0,0.0,256.123993,0.000000,...,95.189293,0.0,0.0,0.000000,0.0,18.165180,0.000000,0.0,0.000000,47.573582
145475,0.000000,0.0,0.000000,0.0,100.995560,0.0,0.0,0.0,0.000000,0.000000,...,128.182999,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000
145476,0.000000,0.0,0.000000,0.0,166.593185,0.0,0.0,0.0,0.000000,0.000000,...,0.000000,0.0,0.0,0.000000,0.0,71.808754,0.000000,0.0,0.000000,49.722225
145477,125.063240,0.0,0.000000,0.0,53.738243,0.0,0.0,0.0,87.231964,0.000000,...,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,66.618019,0.000000


In [ ]:
pd.DataFrame(integration_pred).to_csv("integration_pred_new_method_gex_to_TCR_alpha_chain_10X_all_peptides_225.csv", index=False)